# Sec 2c — Z reconstruction (one-step) group metrics

Chapter plan figures:
* fig_q1_recon_raincloud — Pooled LFP Z decoding quality (DBS-OFF/ON raincloud)
* fig_q1_recon_decomp — Signal decomposition: amplitude r / inst-freq r / phase PLV per model (LFP Z)
* fig_latent_X1X2 — PSID X1 vs X2 subspace contribution to Z reconstruction
* fig_q2_recon_raincloud — Pooled behavioral Z decoding quality (DBS-OFF/ON raincloud)
* fig_q2_recon_decomp — Signal decomposition: amplitude r / inst-freq r / phase PLV per model (behavioral Z)

Appendix:
* fig_076 — Band-grouped reconstruction (theta/alpha/beta/gamma), LFP Z + ECoG Y
* fig_078 — Per-channel mean r, LFP Z, neural mode
* fig_079 — Per-channel mean r, ECoG Y, neural mode
* fig_076b/077 — Raw vs env per-session, neural/behavioral mode
* fig_080/081/080b — Hilbert amplitude r + PLV per-session

In [ ]:
import sys, os
os.chdir("/home/bobby/repos/latent-neural-dynamics-modeling")
sys.path.insert(0, ".")
sys.path.insert(0, "notebooks")

from pathlib import Path

OUT = Path("thesis_figures/sec2")
OUT.mkdir(parents=True, exist_ok=True)
results_root = Path("results").resolve()

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

from modules.style import apply_modules.style, panel_label, COLOR_DBS_OFF, COLOR_DBS_ON, COLOR_PSID, COLOR_DPAD, COLOR_VARMA
from modules.loaders import EXP_BEHAVIORAL, EXP_NEURAL, SESSIONS, load_split_results_required
from modules.utils import normalize_stim, trial_metric_y_for_model, trial_metric_z_for_model
from modules.sec2_common import (
    MODELS, MODEL_COLS, BAND_ORDER, CH_TYPES, DECOMP_METRICS,
    SessionObj, make_session_obj, load_channel_names,
    collect_per_cell_metric, collect_decomp_metrics,
    collect_rawenv_metrics_per_session, collect_hilbert_metrics_per_session,
    collect_band_grouped_metrics, collect_per_feature_means,
    collect_x1x2, x1x2_fig,
    pool_dbs_cells, raincloud_slot,
    dbs_raincloud_fig, decomp_fig, band_raincloud_fig,
    hilbert_raincloud_sessions_fig, rawenv_raincloud_sessions_fig, per_feature_bar_fig,
)

def recon_loader(variant, run_ts, split):
    return load_split_results_required(results_root, variant, run_ts, split)

all_session_objs     = [make_session_obj(results_root, s, EXP_BEHAVIORAL) for s in SESSIONS]
all_lap_session_objs = [make_session_obj(results_root, s, EXP_NEURAL)     for s in SESSIONS]

apply_modules.style()

## Section A — LFP Z reconstruction (neural mode, EXP_NEURAL)

Chapter plan figures (fig_q1_*) and appendix figures for neural mode.
Z target = top-8 Laplacian LFP. Y = ECoG (appendix self-reconstruction).

In [ ]:
lap_cells = [t.label for t in all_lap_session_objs]
lap_z_r = collect_per_cell_metric(
    all_lap_session_objs, recon_loader, trial_metric_z_for_model
)
lap_z_n = collect_per_cell_metric(
    all_lap_session_objs, recon_loader, trial_metric_z_for_model, metric="rmse"
)

In [ ]:
lap_z_r_pool = pool_dbs_cells(lap_z_r, lap_cells)
lap_z_n_pool = pool_dbs_cells(lap_z_n, lap_cells)
fig = dbs_raincloud_fig(lap_z_r_pool, lap_z_n_pool)
fig.savefig(str(OUT / "fig_q1_recon_raincloud.png"))
plt.show()

## fig_q1_recon_decomp — Signal decomposition: amplitude / inst-freq / PLV (LFP Z)

3 rows x 3 cols: rows = amplitude r / inst-freq r / phase PLV, cols = PSID / DPAD / VARMA.
Pooled across all sessions and all raw LFP channels. DBS states pooled (overview figure).

In [ ]:
ch_z_neu = lambda s: load_channel_names(results_root, s, EXP_NEURAL, "Z_features")
q1_decomp = collect_decomp_metrics(
    all_lap_session_objs, EXP_NEURAL, "Z", "test", recon_loader, ch_z_neu
)
fig = decomp_fig(q1_decomp)
fig.savefig(str(OUT / "fig_q1_recon_decomp.png"))
plt.show()

## fig_latent_X1X2 — PSID X1 vs X2 subspace contribution (LFP Z)

Reconstruct Z using full Xp, X1-only (first n1 dims), and X2-only (remaining dims).
Shows which subspace drives LFP reconstruction. Requires HDF5 model artifacts.
One panel per session: 3 grouped bars (full / X1 / X2), coloured by DBS state averaged.

In [ ]:
x1x2_data = collect_x1x2(all_lap_session_objs, results_root, EXP_NEURAL)
fig = x1x2_fig(x1x2_data, SESSIONS)
fig.savefig(str(OUT / "fig_latent_X1X2.png"))
plt.show()

## Fig 76 — Laplacian LFP reconstruction by signal component

Three separate metrics pooled across 4 sessions and all channels within each frequency band:
- **A** Oscillatory reconstruction (`_raw` channels) — correlation coefficient r by band
- **B** Amplitude envelope reconstruction (`_env` channels) — r by band
- **C** Phase coherence (`_raw` channels, Hilbert PLV) by band

In [ ]:
ch_z = lambda s: load_channel_names(results_root, s, EXP_NEURAL, "Z_features")
ch_y = lambda s: load_channel_names(results_root, s, EXP_NEURAL, "Y_features")
lap_corr_raw_z, lap_corr_env_z, lap_plv_z = collect_band_grouped_metrics(
    all_lap_session_objs, EXP_NEURAL, "Z", "test", recon_loader, ch_z
)
lap_corr_raw_y, lap_corr_env_y, lap_plv_y = collect_band_grouped_metrics(
    all_lap_session_objs, EXP_NEURAL, "Y", "test", recon_loader, ch_y
)

fig, axes = plt.subplots(2, 3, figsize=(12, 7.0), layout="constrained")
band_raincloud_fig(axes[0, 0], lap_corr_raw_z, "r", legend=True)
band_raincloud_fig(axes[0, 1], lap_corr_env_z, "r")
band_raincloud_fig(axes[0, 2], lap_plv_z, "PLV", ref_line=None)
panel_label(axes[0, 0], "A", "Laplacian LFP - oscillatory reconstruction")
panel_label(axes[0, 1], "B", "Laplacian LFP - amplitude envelope")
panel_label(axes[0, 2], "C", "Laplacian LFP - phase coherence")
band_raincloud_fig(axes[1, 0], lap_corr_raw_y, "r")
band_raincloud_fig(axes[1, 1], lap_corr_env_y, "r")
band_raincloud_fig(axes[1, 2], lap_plv_y, "PLV", ref_line=None)
panel_label(axes[1, 0], "D", "ECoG - oscillatory reconstruction")
panel_label(axes[1, 1], "E", "ECoG - amplitude envelope")
panel_label(axes[1, 2], "F", "ECoG - phase coherence")
fig.savefig(str(OUT / "fig_076_lap_band_zy.png"))
plt.show()

## Fig 78 — Per-channel Z reconstruction (neural mode, 4 sessions)

Median Pearson r per channel across all trials. 12 Z channels per session (6 raw + 6 env, selected by mRMR).
Channels ordered by mRMR rank (as in Z_features). Raw channels first, then env.

In [ ]:
ch_z = lambda s: load_channel_names(results_root, s, EXP_NEURAL, "Z_features")
lap_sess = [t.label for t in all_lap_session_objs]
z_neural_means, z_neural_feats = collect_per_feature_means(
    all_lap_session_objs,
    EXP_NEURAL,
    "Z",
    "pearson",
    "test",
    recon_loader,
    trial_metric_z_for_model,
    ch_z,
)
fig = per_feature_bar_fig(z_neural_means, z_neural_feats, lap_sess)
fig.savefig(str(OUT / "fig_078_z_neural_perfeature.png"))
plt.show()

In [ ]:
ch_y = lambda s: load_channel_names(results_root, s, EXP_NEURAL, "Y_features")
y_neural_means, y_neural_feats = collect_per_feature_means(
    all_lap_session_objs,
    EXP_NEURAL,
    "Y",
    "pearson",
    "test",
    recon_loader,
    trial_metric_y_for_model,
    ch_y,
)
fig = per_feature_bar_fig(y_neural_means, y_neural_feats, lap_sess)
fig.savefig(str(OUT / "fig_079_y_neural_perfeature.png"))
plt.show()

## Section B — ECoG & Tracing Kinematics (behavioral mode, EXP_BEHAVIORAL)

Y = top-12 ECoG self-reconstruction. Z = tracing kinematics (velocity_x + acc_mag, 2 features).
Same figure templates as Section A.

In [ ]:
beh_cells = [t.label for t in all_session_objs]
beh_z_r = collect_per_cell_metric(
    all_session_objs, recon_loader, trial_metric_z_for_model
)
beh_z_n = collect_per_cell_metric(
    all_session_objs, recon_loader, trial_metric_z_for_model, metric="rmse"
)

In [ ]:
beh_z_r_pool = pool_dbs_cells(beh_z_r, beh_cells)
beh_z_n_pool = pool_dbs_cells(beh_z_n, beh_cells)
fig = dbs_raincloud_fig(beh_z_r_pool, beh_z_n_pool)
fig.savefig(str(OUT / "fig_q2_recon_raincloud.png"))
plt.show()

## fig_q2_recon_decomp — Signal decomposition: amplitude / inst-freq / PLV (behavioral Z)

Same 3x3 structure as fig_q1_recon_decomp but for Z = tracing kinematics (velocity_x, acc_mag).
Note: behavioral Z has no band structure; _raw suffix channels still selected for Hilbert.

In [ ]:
ch_z_beh = lambda s: load_channel_names(results_root, s, EXP_BEHAVIORAL, "Z_features")
q2_decomp = collect_decomp_metrics(
    all_session_objs,
    EXP_BEHAVIORAL,
    "Z",
    "test",
    recon_loader,
    ch_z_beh,
    raw_only=False,
)
fig = decomp_fig(q2_decomp)
fig.savefig(str(OUT / "fig_q2_recon_decomp.png"))
plt.show()

## Fig 77 — Y reconstruction: oscillatory (raw) vs amplitude (env) — behavioral mode

Same raw/env split but on Y (ECoG self-reconstruction) in behavioral mode.
Z = behavioral targets (2 channels, no band structure) so not broken down here.

In [ ]:
ch_y_beh = lambda s: load_channel_names(results_root, s, EXP_BEHAVIORAL, "Y_features")
beh_y_r_ps = collect_rawenv_metrics_per_session(
    all_session_objs,
    EXP_BEHAVIORAL,
    "Y",
    "pearson",
    "test",
    recon_loader,
    trial_metric_y_for_model,
    ch_y_beh,
)
beh_y_n_ps = collect_rawenv_metrics_per_session(
    all_session_objs,
    EXP_BEHAVIORAL,
    "Y",
    "rmse",
    "test",
    recon_loader,
    trial_metric_y_for_model,
    ch_y_beh,
)
fig, axes = plt.subplots(4, 2, figsize=(8, 12), layout="constrained")
rawenv_raincloud_sessions_fig(axes, beh_y_r_ps, beh_y_n_ps, SESSIONS)
fig.savefig(str(OUT / "fig_077_beh_rawenv_y_per_session.png"))
plt.show()

In [ ]:
ch_y_neu = lambda s: load_channel_names(results_root, s, EXP_NEURAL, "Y_features")
neural_y_r_ps = collect_rawenv_metrics_per_session(
    all_lap_session_objs,
    EXP_NEURAL,
    "Y",
    "pearson",
    "test",
    recon_loader,
    trial_metric_y_for_model,
    ch_y_neu,
)
neural_y_n_ps = collect_rawenv_metrics_per_session(
    all_lap_session_objs,
    EXP_NEURAL,
    "Y",
    "rmse",
    "test",
    recon_loader,
    trial_metric_y_for_model,
    ch_y_neu,
)
fig, axes = plt.subplots(4, 2, figsize=(9, 7), layout="constrained")
rawenv_raincloud_sessions_fig(axes, neural_y_r_ps, neural_y_n_ps, SESSIONS)
fig.savefig(str(OUT / "fig_076b_neural_rawenv_y_per_session.png"))
plt.show()

## Fig 79 — Per-channel Y reconstruction (behavioral mode, 4 sessions)

Same bar layout but for Y (ECoG self-reconstruction) in behavioral mode.
12 ECoG channels per session (6 raw + 6 env), mRMR-selected.

In [ ]:
ch_y_beh = lambda s: load_channel_names(results_root, s, EXP_BEHAVIORAL, "Y_features")
beh_sess = [t.label for t in all_session_objs]
y_beh_means, y_beh_feats = collect_per_feature_means(
    all_session_objs,
    EXP_BEHAVIORAL,
    "Y",
    "pearson",
    "test",
    recon_loader,
    trial_metric_y_for_model,
    ch_y_beh,
)
fig = per_feature_bar_fig(y_beh_means, y_beh_feats, beh_sess)
fig.savefig(str(OUT / "fig_079b_y_beh_perfeature.png"))
plt.show()

In [ ]:
ch_z_beh = lambda s: load_channel_names(results_root, s, EXP_BEHAVIORAL, "Z_features")
z_beh_means, z_beh_feats = collect_per_feature_means(
    all_session_objs,
    EXP_BEHAVIORAL,
    "Z",
    "pearson",
    "test",
    recon_loader,
    trial_metric_z_for_model,
    ch_z_beh,
)
fig = per_feature_bar_fig(z_beh_means, z_beh_feats, beh_sess)
fig.savefig(str(OUT / "fig_079c_z_beh_perfeature.png"))
plt.show()

## Fig 80-81 — Hilbert amplitude vs phase coherence (raw channels only)

For each raw channel (`_raw` suffix), apply Hilbert transform to both true and predicted signals:
- **Amplitude r**: Pearson r between |H(true)| and |H(pred)| — does model track amplitude envelope?
- **Phase PLV**: phase-locking value |mean(exp(i*(phase_true - phase_pred)))| — does model track instantaneous phase?

Fig 80: Z raw channels, neural mode. Fig 81: Y raw channels (ECoG), behavioral mode.

In [ ]:
ch_z_neu = lambda s: load_channel_names(results_root, s, EXP_NEURAL, "Z_features")
neural_z_hilbert_ps = collect_hilbert_metrics_per_session(
    all_lap_session_objs, EXP_NEURAL, "Z", "test", recon_loader, ch_z_neu
)
fig, axes = plt.subplots(4, 2, figsize=(8, 12), layout="constrained")
hilbert_raincloud_sessions_fig(axes, neural_z_hilbert_ps, SESSIONS)
fig.savefig(str(OUT / "fig_080_lap_hilbert_z_per_session.png"))
plt.show()

In [ ]:
ch_y_beh = lambda s: load_channel_names(results_root, s, EXP_BEHAVIORAL, "Y_features")
beh_y_hilbert_ps = collect_hilbert_metrics_per_session(
    all_session_objs, EXP_BEHAVIORAL, "Y", "test", recon_loader, ch_y_beh
)
fig, axes = plt.subplots(4, 2, figsize=(8, 12), layout="constrained")
hilbert_raincloud_sessions_fig(axes, beh_y_hilbert_ps, SESSIONS)
fig.savefig(str(OUT / "fig_081_beh_hilbert_y_per_session.png"))
plt.show()

In [ ]:
ch_y_neu = lambda s: load_channel_names(results_root, s, EXP_NEURAL, "Y_features")
neural_y_hilbert_ps = collect_hilbert_metrics_per_session(
    all_lap_session_objs, EXP_NEURAL, "Y", "test", recon_loader, ch_y_neu
)
fig, axes = plt.subplots(4, 2, figsize=(9, 7), layout="constrained")
hilbert_raincloud_sessions_fig(axes, neural_y_hilbert_ps, SESSIONS)
fig.savefig(str(OUT / "fig_080b_neural_hilbert_y_per_session.png"))
plt.show()